<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/uni_mol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install rdkit
!pip install unimol_tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.9/120.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 17.3 MB/s eta 0:00:00


In [8]:
import os
import torch
import numpy as np
from rdkit import Chem
from unimol_tools import UniMolRepr
import warnings
warnings.filterwarnings('ignore')

SDF_BASE_DIR = "/content/CV_Folds"
OUTPUT_DIR = "./data/embeddings"
FOLDS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Loading Uni-Mol...")
unimol = UniMolRepr(data_type='molecule', remove_hs=False)

def process_fold_files(active_sdf_path, negative_sdf_path, output_name):
    """Reads both active and negative SDFs, combines them, and generates Uni-Mol embeddings."""

    # Initialize as a dictionary of lists
    custom_data = {
        "atoms": [],
        "coordinates": []
    }
    labels = []

    # Helper function to read an SDF and assign a fixed label based on the file source
    def read_sdf(sdf_path, assigned_label):
        if not os.path.exists(sdf_path):
            print(f"  -> Warning: File not found: {sdf_path}")
            return

        supplier = Chem.SDMolSupplier(sdf_path, removeHs=False, sanitize=True)
        for mol in supplier:
            if mol is None:
                continue

            labels.append(assigned_label)
            conf = mol.GetConformer()
            atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
            coords = conf.GetPositions().tolist()

            # FIX: Append to the respective lists in the dictionary
            custom_data["atoms"].append(atoms)
            custom_data["coordinates"].append(coords)

    # Read actives (Label 1.0) and negatives (Label 0.0)
    read_sdf(active_sdf_path, 1.0)
    read_sdf(negative_sdf_path, 0.0)

    if len(labels) == 0:
        print(f"No valid molecules found for {output_name}. Skipping.")
        return

    print(f"Extracting representations for {output_name} ({len(labels)} compounds)...")

  # Get representations
    repr_output = unimol.get_repr(custom_data, return_atomic_reprs=False)


    # This check extracts the [CLS] embeddings regardless of the structure.
    if isinstance(repr_output, dict):
        # Case 1: It returned a dictionary
        cls_embeddings = np.array(repr_output['cls_repr'])

    elif isinstance(repr_output, list) or isinstance(repr_output, np.ndarray):
        if len(repr_output) > 0 and isinstance(repr_output[0], dict):
            # Case 2: It returned a list of dictionaries
            cls_embeddings = np.array([item['cls_repr'] for item in repr_output])
        else:
            # Case 3: It returned the list of embeddings directly
            cls_embeddings = np.array(repr_output)

    else:
        raise ValueError(f"Unexpected output type from Uni-Mol: {type(repr_output)}")

    # Check shape to ensure it matches (Num_Molecules, Hidden_Dimension)
    print(f"Extracted embeddings shape: {cls_embeddings.shape}")

    # Convert to PyTorch tensors
    tensor_embeddings = torch.tensor(cls_embeddings, dtype=torch.float32)
    tensor_labels = torch.tensor(labels, dtype=torch.float32)

    # Save the combined tensor file
    out_file = os.path.join(OUTPUT_DIR, f"{output_name}.pt")
    torch.save({"embeddings": tensor_embeddings, "labels": tensor_labels}, out_file)
    print(f"Saved {tensor_embeddings.shape[0]} compounds to {out_file}\n")

if __name__ == "__main__":
    for i in range(1, FOLDS + 1):
        print(f"Processing Fold {i}: ")

        # Define paths based on your specific naming convention
        train_actives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Train_Actives_3D.sdf")
        train_negatives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Train_Negatives_3D.sdf")

        val_actives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Val_Actives_3D.sdf")
        val_negatives = os.path.join(SDF_BASE_DIR, f"Fold_{i}_Val_Negatives_3D.sdf")

        # Process and combine Train set
        process_fold_files(train_actives, train_negatives, f"Fold_{i}_Train")

        # Process and combine Val set
        process_fold_files(val_actives, val_negatives, f"Fold_{i}_Val")

print("All Uni-Mol embeddings generated successfully!")

INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[PAD]', index is 2.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[CLS]', index is 0.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[SEP]', index is 3.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[UNK]', index is 1.


Loading Uni-Mol...


2026-07-10 22:19:12 | unimol_tools/models/unimol.py | 169 | INFO | Uni-Mol Tools | Loading pretrained weights from /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/mol_pre_all_h_220816.pt
INFO:Uni-Mol Tools:Loading pretrained weights from /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/mol_pre_all_h_220816.pt


--- Processing Fold 1 ---


INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[PAD]', index is 2.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[CLS]', index is 0.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[SEP]', index is 3.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[UNK]', index is 1.
2026-07-10 22:19:13 | unimol_tools/tasks/trainer.py | 103 | INFO | Uni-Mol Tools | Using CPU.
INFO:Uni-Mol Tools:Using CPU.


...

Extracting representations for Fold_5_Train (550 compounds)...


100%|██████████| 18/18 [01:11<00:00,  3.97s/it]
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[PAD]', index is 2.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[CLS]', index is 0.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[SEP]', index is 3.
INFO:unimol_tools.data.dictionary:Duplicate word found when loading Dictionary: '[UNK]', index is 1.
2026-07-10 22:26:24 | unimol_tools/tasks/trainer.py | 103 | INFO | Uni-Mol Tools | Using CPU.
INFO:Uni-Mol Tools:Using CPU.


Extracted embeddings shape: (550, 512)
Saved 550 compounds to ./data/embeddings/Fold_5_Train.pt

Extracting representations for Fold_5_Val (135 compounds)...


100%|██████████| 5/5 [00:16<00:00,  3.29s/it]

Extracted embeddings shape: (135, 512)
Saved 135 compounds to ./data/embeddings/Fold_5_Val.pt

All Uni-Mol embeddings generated successfully!


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.1):
        super(FocalLossWithSmoothing, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        targets = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return torch.mean(focal_weight * BCE_loss)

class UniMolMLPHead(nn.Module):
    def __init__(self, input_dim=512, hidden_dim_1=256, hidden_dim_2=64, dropout_rate=0.5):
        super(UniMolMLPHead, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.BatchNorm1d(hidden_dim_1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.BatchNorm1d(hidden_dim_2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_2, 1)
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x).squeeze(-1)



Starting Fold 1/5


ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 256])

In [ ]:
import os
import torch
from torch.utils.data import Dataset

class UniMolFoldDataset(Dataset):
    def __init__(self, fold_idx, is_train, base_dir="./data/embeddings"):
        phase = "Train" if is_train else "Val"
        file_path = os.path.join(base_dir, f"Fold_{fold_idx}_{phase}.pt")
        data = torch.load(file_path)
        self.embeddings = data["embeddings"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, matthews_corrcoef

from dataset import UniMolFoldDataset
from models import UniMolMLPHead, FocalLossWithSmoothing

def train_fold(fold_idx, epochs=150, batch_size=32, lr=5e-4):
    print(f"\n{'='*40}\nStarting Fold {fold_idx}/5\n{'='*40}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = UniMolFoldDataset(fold_idx, is_train=True)
    val_dataset = UniMolFoldDataset(fold_idx, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = UniMolMLPHead(input_dim=512).to(device)
    criterion = FocalLossWithSmoothing(alpha=0.75, gamma=2.0, smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

    best_mcc = -1.0
    best_auc = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for embeddings, labels in train_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(embeddings)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * embeddings.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        all_logits = []
        all_labels = []

        with torch.no_grad():
            for embeddings, labels in val_loader:
                embeddings, labels = embeddings.to(device), labels.to(device)
                logits = model(embeddings)
                loss = criterion(logits, labels)
                val_loss += loss.item() * embeddings.size(0)
                all_logits.extend(logits.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader.dataset)

        probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
        preds = (probs > 0.5).astype(int)

        try:
            auc = roc_auc_score(all_labels, probs)
            mcc = matthews_corrcoef(all_labels, preds)
        except ValueError:
            auc, mcc = 0.0, 0.0

        scheduler.step(mcc)

        if mcc > best_mcc:
            best_mcc = mcc
            best_auc = auc
            torch.save(model.state_dict(), f"best_model_fold_{fold_idx}.pth")

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {auc:.4f} | Val MCC: {mcc:.4f}")

    print(f"Fold {fold_idx} Completed. Best Validation MCC: {best_mcc:.4f} | Associated AUC: {best_auc:.4f}")
    return best_auc, best_mcc


In [ ]:
import numpy as np
from engine import train_fold

if __name__ == "__main__":
    cv_aucs = []
    cv_mccs = []

    for i in range(1, 6):
        auc, mcc = train_fold(fold_idx=i, epochs=150, batch_size=32, lr=5e-4)
        cv_aucs.append(auc)
        cv_mccs.append(mcc)

    print("\n" + "="*40)
    print("5-FOLD CROSS VALIDATION RESULTS (3D Uni-Mol 2 Base)")
    print("="*40)
    print(f"Average ROC-AUC : {np.mean(cv_aucs):.4f} +/- {np.std(cv_aucs):.4f}")
    print(f"Average MCC     : {np.mean(cv_mccs):.4f} +/- {np.std(cv_mccs):.4f}")
